In [1]:
import pandas as pd

# Cargar y ordenar los 273 modelos por Recall Medio decreciente
df_all = pd.read_csv("01_todos_los_modelos_consolidados.csv", sep=";")
df_all_sorted = df_all.sort_values(by="Recall Medio", ascending=False).reset_index(drop=True)

def formatear_cfg(c):
    try:
        d = eval(c)
        if isinstance(d, dict):
            return f"n={d.get('n_estimators')}, lr={d.get('learning_rate')}, d={d.get('max_depth')}, s={d.get('subsample')}"
    except:
        pass
    return c.replace('_', r'\_')

with open("tabla_todos_los_modelos_3f.tex", "w", encoding="utf-8") as f:
    f.write(r"""\begin{longtable}{|l|p{3.8cm}|c|c|c|c|c|c|}
\caption{Comparativa exhaustiva de los 273 modelos evaluados (3 baselines, 135 XGBoost y 135 LightGBM) en validación cruzada estratificada ($K=5$), ordenados por Recall medio decreciente.} \label{tab:todos_modelos_exhaustiva} \\
\hline
\rowcolor{gray!25} \textbf{Familia} & \textbf{Configuración} & \textbf{Recall} & \textbf{Espec.} & \textbf{Prec.} & \textbf{$F_1$} & \textbf{AUC-ROC} & \textbf{AUC-PR} \\ \hline
\endfirsthead
\multicolumn{8}{c}{{\bfseries \tablename\ \thetable{} -- Continuación de la página anterior}} \\
\hline
\rowcolor{gray!25} \textbf{Familia} & \textbf{Configuración} & \textbf{Recall} & \textbf{Espec.} & \textbf{Prec.} & \textbf{$F_1$} & \textbf{AUC-ROC} & \textbf{AUC-PR} \\ \hline
\endhead
\hline \multicolumn{8}{|r|}{{Continúa en la siguiente página...}} \\ \hline
\endfoot
\hline
\endlastfoot
""")
    for _, r in df_all_sorted.iterrows():
        fam = r['Familia']
        cfg = formatear_cfg(str(r['Configuracion / Hiperparametros']))
        rec = f"{r['Recall Medio']:.3f} $\\pm$ {r['Recall Std']:.3f}"
        esp = f"{r['Especificidad Media']:.3f} $\\pm$ {r['Especificidad Std']:.3f}" if pd.notna(r['Especificidad Media']) else "-"
        prec = f"{r['Precision Media']:.3f} $\\pm$ {r['Precision Std']:.3f}"
        f1 = f"{r['F1-Score Medio']:.3f} $\\pm$ {r['F1-Score Std']:.3f}" if pd.notna(r['F1-Score Medio']) else "-"
        roc = f"{r['AUC-ROC Medio']:.3f} $\\pm$ {r['AUC-ROC Std']:.3f}"
        pr = f"{r['AUC-PR Medio']:.3f} $\\pm$ {r['AUC-PR Std']:.3f}" if pd.notna(r['AUC-PR Medio']) else "-"
        f.write(f"{fam} & {cfg} & {rec} & {esp} & {prec} & {f1} & {roc} & {pr} \\\\ \\hline\n")
    f.write(r"\end{longtable}" + "\n")

print(f"✅ 'tabla_todos_los_modelos_3f.tex' generada con éxito con sus {len(df_all_sorted)} modelos.")

✅ 'tabla_todos_los_modelos_3f.tex' generada con éxito con sus 273 modelos.


In [ ]:
import ast
import os
import pandas as pd

csv_file = "01_todos_los_modelos_consolidados.csv"
if not os.path.exists(csv_file):
  for f in os.listdir("."):
    if "consolidados" in f and f.endswith(".csv"):
      csv_file = f
      break

df = pd.read_csv(csv_file, sep=";")

df_sorted = df.sort_values(by="Recall Medio", ascending=False).reset_index(
    drop=True
)


def formatear_cfg_centrada(cfg_str: str) -> str:
  try:
    d = ast.literal_eval(cfg_str)
    if isinstance(d, dict):
      n = d.get("n_estimators", "-")
      lr = d.get("learning_rate", "-")
      depth = d.get("max_depth", "-")
      s = d.get("subsample", "-")
      return f"n={n}, lr={lr}\\newline d={depth}, s={s}"
  except (ValueError, SyntaxError):
    pass

  if "most_frequent" in cfg_str:
    return "strategy=\\newline\\texttt{'most\\_frequent'}"
  if "balanced" in cfg_str and "1000" in cfg_str:
    return "class\\_weight='bal.',\\newline max\\_iter=1000"
  if "depth=5" in cfg_str:
    return "max\\_depth=5,\\newline class\\_weight='bal.'"

  return cfg_str.replace("_", r"\_")


def pmval(mean: float, std: float) -> str:
  if pd.isna(mean):
    return "-"
  # Formato directo en dos líneas con \newline y \tiny para el centrado vertical con m{...}
  return f"{mean:.4f}\\newline{{\\tiny $\\pm${std:.4f}}}"


lineas_latex = []
for _, r in df_sorted.iterrows():
  fam = r["Familia"]
  cfg = formatear_cfg_centrada(str(r["Configuracion / Hiperparametros"]))
  rec = pmval(r["Recall Medio"], r["Recall Std"])
  esp = pmval(r["Especificidad Media"], r["Especificidad Std"])
  prec = pmval(r["Precision Media"], r["Precision Std"])
  f1 = pmval(r["F1-Score Medio"], r["F1-Score Std"])
  roc = pmval(r["AUC-ROC Medio"], r["AUC-ROC Std"])
  pr = pmval(r["AUC-PR Medio"], r["AUC-PR Std"])

  linea = (
      f"{fam} & {cfg} & {rec} & {esp} & {prec} & {f1} & {roc} & {pr} \\\\ \\hline"
  )
  lineas_latex.append(linea)

output_file = "lineas_tabla_modelos.tex"
with open(output_file, "w", encoding="utf-8") as f:
  f.write("\n".join(lineas_latex) + "\n")

print(f"✅ Se han generado las {len(lineas_latex)} líneas en '{output_file}'.")

✅ Se han generado las 273 líneas en 'lineas_tabla_modelos.tex'.


: 